In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv('data/cleaned_data.csv')
data.head()

,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,fat_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,Banana Chips Sweetened (Whole),not mentioned,NaN,Labels are missing,"Bananas, vegetable oil (coconut oil, corn oil ...",unknown,No additives,d,2243.0,28.57,...,0.0,no information,14.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Peanuts,torn & glasser,NaN,Labels are missing,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:wheat, en:soy, en:peanuts",No additives,b,1941.0,17.86,...,0.0,no information,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Organic Salted Nut Mix,grizzlies,NaN,Labels are missing,"Organic hazelnuts, organic cashews, organic wa...",unknown,No additives,d,2540.0,57.14,...,0.0,no information,12.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Organic Polenta,bob's red mill,NaN,Labels are missing,Organic polenta,unknown,No additives,not given,1552.0,1.43,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN
4,Breadshop Honey Gone Nuts Granola,unfi,NaN,Labels are missing,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,18.27,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
us_data = pd.read_csv('data/us_data.csv')
us_data.shape

(171521, 98)

In [9]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].apply(clean_ingredients)
us_data['ingredients'] = us_data['ingredients_text'].apply(clean_ingredients)

In [10]:
data = data.drop(columns=['ingredients_text'])
us_data = us_data.drop(columns=['ingredients_text'])

### Pre-processing 

In [11]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['product_name'] = data['product_name'].apply(clean_text)
data['ingredients'] = data['ingredients'].apply(clean_text)
data['allergens_en'] = data['allergens_en'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

In [12]:
# Replace placeholders with NaN or empty lists
data['allergens_en'] = data['allergens_en'].replace('unknown', np.nan)
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)

In [13]:
# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')
data['allergens_en'] = data['allergens_en'].str.split(', ')

### Implement product matching

In [14]:
from fuzzywuzzy import process

def find_top_matches(user_input, choices, limit=5):
    matches = process.extract(user_input, choices, limit=limit)
    return matches

In [ ]:
# Example: User inputs a product name
user_input = "Protein bar"
top_matches = find_top_matches(user_input, data['product_name'].tolist())

print(f"Top matches for '{user_input}':")
for match, score in top_matches:
    print(f"- {match} (Score: {score})")

### Category filtering

In [15]:
def get_primary_category(categories_en):
    if pd.isna(categories_en):
        return None
    # Split the hierarchy by commas and get the last part
    categories = categories_en.split(',')
    return categories[-1].strip()  # Return the last category (primary category)

def find_primary_category_from_matches(top_matches, df):
    for match, score in top_matches:
        categories_en = df[df['product_name'] == match]['categories_en'].values[0]
        primary_category = get_primary_category(categories_en)
        if primary_category is not None:
            return primary_category, match
    return None, None  # If no valid category is found

### Allergens filtering

In [16]:
def filter_by_allergens(products, allergens_to_avoid):
    for allergen in allergens_to_avoid:
        if f'contains_{allergen}' in products.columns:
            products = products[~products[f'contains_{allergen}']]
    return products

### Recommendations

In [17]:
def generate_recommendations(filtered_products, top_n=5):
    return filtered_products[['product_name', 'additives_en']].head(top_n)

In [18]:
def recommend_products(user_input, allergens_to_avoid, df, top_n=5):
    # Step 1: Find top 5 closest matches
    top_matches = find_top_matches(user_input, df['product_name'].tolist())
    
    # Step 2: Find primary category from top matches
    primary_category, matched_product = find_primary_category_from_matches(top_matches, df)
    
    if primary_category is not None:
        # Step 3: Filter products in the primary category
        same_category_products = df[df['categories_en'].str.contains(primary_category, case=False, na=False)]
        
        # Step 4: Filter by allergens
        filtered_products = filter_by_allergens(same_category_products, allergens_to_avoid)
        
        # Step 5: Generate recommendations
        recommendations = generate_recommendations(filtered_products, top_n)
    else:
        # Fallback: Suggest closest matches based on product name similarity
        print("Warning: No valid category found in top matches. Suggesting closest matches.")
        recommendations = df[df['product_name'].isin([match[0] for match in top_matches])][['product_name', 'additives_en']].head(top_n)
    
    return recommendations 

In [19]:
# Example usage
user_input = "Protein bar"  # User's input product name
allergens_to_avoid = ['soy', 'peanuts']  # Allergens to avoid
recommendations = recommend_products(user_input, allergens_to_avoid, us_data)

# Print recommendations
print("Final recommendations:")
print(recommendations)

Final recommendations:
       product_name           additives_en
105289  Protein Bar             E322,E322i
113422  Protein Bar                   E968
114611  Protein Bar  E322,E322i,E965,E965i
114614  Protein Bar  E322,E322i,E965,E965i
114616  Protein Bar  E322,E322i,E965,E965i


### Problem

Now, what are we gonna do for the products that has no category values?

### Radial chart

In [ ]:
# Group by category_level_1 and calculate the mean of nutritional columns
macro_nutrition = ['energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fiber_100g', 'fat_100g']
category_nutrition = us_data.groupby('category_level_1')[macro_nutrition].mean().reset_index()

In [ ]:
# Get the top 10 categories by product count
top_categories = us_data['category_level_1'].value_counts().nlargest(10).index
top_category_nutrition = category_nutrition[category_nutrition['category_level_1'].isin(top_categories)]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalize the nutritional columns
scaler = MinMaxScaler()
top_category_nutrition[macro_nutrition] = scaler.fit_transform(top_category_nutrition[macro_nutrition])

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Function to create an interactive radar chart
def create_interactive_radar_chart(categories, values, title):
    fig = go.Figure()

    for i, category in enumerate(categories):
        fig.add_trace(go.Scatterpolar(
            r=values.iloc[i].tolist(),  # Nutritional values
            theta=values.columns,       # Nutritional metrics
            fill='toself',              # Fill the area under the line
            name=category               # Category name
        ))

    # Update layout for better visualization
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]  # Normalized scale
            )
        ),
        title=title,
        showlegend=True
    )

    # Show the chart
    fig.show()

# Prepare data for the radar chart
categories = top_category_nutrition['category_level_1']
values = top_category_nutrition[macro_nutrition]

In [ ]:
# Function to create a radar chart with a dropdown
def create_radar_chart_with_dropdown(categories, values, title):
    fig = go.Figure()

    for i, category in enumerate(categories):
        fig.add_trace(go.Scatterpolar(
            r=values.iloc[i].tolist(),
            theta=values.columns,
            fill='toself',
            name=category,
            visible=True  # Make all traces visible by default
        ))

    # Create dropdown options
    dropdown_options = []
    for i, category in enumerate(categories):
        dropdown_options.append(
            dict(
                args=[{"visible": [j == i for j in range(len(categories))]}],
                label=category,
                method="update"
            )
        )

    # Add dropdown to the layout
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=dropdown_options,
                direction="down",
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            )
        ],
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        ),
        title=title,
        showlegend=True
    )

    # Show the chart
    fig.show()

# Create the interactive radar chart with a dropdown
create_radar_chart_with_dropdown(categories, values, title='Top 10 Primary Categories by Nutritional Facts')

In [ ]:
us_data.columns.values